# 🏗️ LLM 推理服务架构全景

**目标**：从零开始系统理解大模型推理服务的技术栈——从 GGUF 量化到 PagedAttention，从单卡本地到分布式集群。

> 💡 **新手？** 如果对 token、自回归生成、KV Cache、量化这些概念还不熟，建议先读 `01-01-theory/` 下的六篇前置理论，再回来看这篇 overview。推荐顺序：**01-theory/ → 02-frameworks/ → 03-models/**。

## 为什么需要专门的推理框架？

直接用 PyTorch `model.generate()` 跑推理的问题：

| 瓶颈 | 现象 | 根因 |
|------|------|------|
| **显存碎片** | 并发请求时 OOM，但显存实际有剩余 | KV Cache 动态分配无回收机制 |
| **低吞吐** | GPU 利用率 < 30% | 静态 batching — 等最长的请求完成才能换下一批 |
| **延迟抖动** | P99 延迟是 P50 的 10x+ | 无抢占调度，长序列阻塞短序列 |
| **量化低效** | 4-bit 量化精度损失大 | 简单 round-to-nearest，不考虑权重的 outlier 分布 |

## 推理服务的核心问题域

```
┌──────────────────────────────────────────────────────────────┐
│                      推理服务架构全景                          │
├──────────────┬───────────────────┬──────────────────────────┤
│   本地推理     │    服务端高吞吐     │      分布式推理            │
│  (Local)      │  (Server)          │  (Distributed)           │
├──────────────┼───────────────────┼──────────────────────────┤
│ llama.cpp     │ vLLM               │ SGLang                    │
│ Ollama        │ TensorRT-LLM       │ Ray Serve + LLM           │
│ LM Studio     │ TGI (HuggingFace)  │ vLLM multi-node           │
│               │                    │ DeepSpeed-MII             │
├──────────────┼───────────────────┼──────────────────────────┤
│ 关键需求：     │ 关键需求：           │ 关键需求：                 │
│ • 消费级 GPU   │ • 最大化吞吐         │ • 模型太大单卡装不下       │
│ • 无 GPU 也行  │ • 低延迟 SLA         │ • 需要弹性伸缩             │
│ • 隐私/离线    │ • 高并发             │ • 异构硬件混合             │
│ • 易用性       │ • 抢占式调度         │ • 容错                     │
└──────────────┴───────────────────┴──────────────────────────┘
```

## 核心技术维度对比

### 1. Attention 与 KV Cache 管理

这是推理框架最大的分水岭。

| 框架 | KV Cache 策略 | 核心创新 | 显存利用率 |
|------|-------------|---------|----------|
| **llama.cpp** | 预分配连续 buffer，按最大 sequence length 分配 | GGUF 量化直接应用于 KV Cache | 低（内部碎片） |
| **Ollama** | 继承 llama.cpp，加一层 LRU 淘汰 | 请求级 KV Cache 复用 | 同 llama.cpp |
| **vLLM** | **PagedAttention** — 分块管理，block table 映射 | 类比操作系统虚拟内存，按需分配 + 共享 | **高（~4x 提升）** |
| **TensorRT-LLM** | 类似 PagedAttention 的 block 管理 | 加上 **in-flight batching**（不等整批完成） | 高 |
| **SGLang** | **RadixAttention** — 基于 radix tree 的前缀缓存 | 自动发现和复用公共前缀（如 system prompt） | 最高（前缀复用场景） |
| **TGI** | 静态预分配 + 可选的 PagedAttention 后端 | 最早提出 prefix caching（但实现较简单） | 中 |

### 2. 调度策略

```
静态 Batching（PyTorch 原生）：
  Batch 0: [req0 ████████████████████]  ← 等最长的 req0 完成
           [req1 ████]                   ← req1 早就完了，GPU 空闲等待
  Batch 1: [req2 ██████████]
           [req3 ████████████████████████████████]
  问题：GPU 计算单元在等 — 类似餐厅等最慢的人吃完才翻台

Continuous Batching（vLLM/TGI）：
  Step 0: [req0 ████] [req1 ██]           ← req0, req1 各生成一个 token
  Step 1: [req0 █████] [req1 ███]          ← req2 到达，插入！
          [req2 █]                        ← 不等批次结束，动态增减
  Step 2: [req0 ██████] [req1 ████]       ← req1 完成，立即释放
          [req2 ██] [req3 █]              ← req3 到达，插入！
  效果：GPU 利用率 30% → 80%+

In-flight Batching（TensorRT-LLM）：
  在 continuous batching 基础上更进一步：
  - 请求可以在 decode 阶段被挂起（context phase 先不处理）
  - prefilling 和 decoding 可以混在同一批次
  - 效果：长 context 请求不会阻塞短请求的解码
```

### 3. 量化方案

| 框架 | 量化引擎 | 代表格式 | 特点 |
|------|---------|---------|------|
| llama.cpp | ggml 内置 | Q4_0, Q4_K_M, Q6_K, IQ4_NL | K-quant 系列是 SOTA 的 CPU 量化 |
| vLLM | 委托给后端 | AWQ, GPTQ (通过 AutoAWQ/AutoGPTQ) | 不自己实现量化，兼容性好 |
| TensorRT-LLM | NVIDIA 原生 | FP8, INT8, INT4 (TensorRT engine) | 硬件感知量化，最高效 |
| SGLang | 同 vLLM | AWQ, GPTQ, FP8 | 支持更多实验性量化方案 |

## 选型决策树

```
需要生产级 API 服务？
├── 是 → 有 NVIDIA GPU？
│        ├── 是 → 需要极致吞吐？
│        │        ├── 是 → TensorRT-LLM（固定模型） / vLLM（灵活模型）
│        │        └── 否 → vLLM（默认选择，生态最好）
│        └── 否 → Apple Silicon？
│                 ├── 是 → llama.cpp + MLX（macOS 原生）
│                 └── 否 → llama.cpp / Ollama（CPU 推理）
└── 否 → 本地使用，隐私优先
         ├── 易用性优先 → Ollama（一键启动）
         ├── 需要 API 兼容 → llama.cpp server（OpenAI 兼容 API）
         └── 极致性能 → llama.cpp 直接调用 + 手写 pipeline
```

## 前置理论

如果你是新手，建议先读 `01-01-theory/` 的六篇基础（按顺序）：

| # | 篇目 | 核心内容 | 为什么先读 |
|---|------|---------|----------|
| 01 | LLM 推理速览 | token、自回归生成、TTFT/TPOT | 建立推理的正确 mental model |
| 02 | **Transformer 架构完整解析** | Self-Attention、Multi-Head、位置编码、Encoder-Decoder | **理论根基**，论文 + 代码双线讲解 |
| 03 | Transformer 推理视角 | Prefill/Decode、KV Cache、显存分解 | 理解所有框架优化的出发点 |
| 04 | 量化基础 | GGUF/AWQ/GPTQ/FP8、KV Cache 量化 | llama.cpp/vLLM/TensorRT-LLM 的共同语言 |
| 05 | GPU 显存布局 | HBM/SRAM、多模态、工具调用、显存管理 | 理解显存为什么是核心瓶颈 |
| 06 | **混合推理优化** | 四大框架的工具调用/多模态优化、prefix caching 对比 | 选型决策 + 生产配置指南 |

> 如果你已经熟悉 Transformer 架构和量化原理，可以直接跳到下面的学习路径。

## 下一步

按学习路径进入各 notebook：

0. **原理** → `01-theory/`
   - 01 → 02 → 03 → 04 → 05 → 06 (按编号顺序)

1. **框架** → `02-frameworks/`
   - `llama-cpp/` — 本地推理、量化、CPU 优化
   - `ollama/` — llama.cpp 之上的易用封装
   - `vllm/` — PagedAttention 与调度（6 篇系列）
   - `sglang/` — RadixAttention 与约束生成（5 篇系列）
   - `tensorrt-llm/` — NVIDIA 编译优化
   - `tgi/` — HuggingFace 生态的推理标准
   - `ray-serve/` — 分布式调度

2. **模型** → `03-models/`
   - `deepseek-v4/` — 混合稀疏注意力、FP4、ShadowRadix（6 篇系列）

每篇 notebook 的结构：
```
架构概览 → 核心数据结构（源码级） → 关键路径走读 → 实验验证 → 对比思考
```

In [ ]:
# 快速验证：检查本机 GPU 情况
import subprocess, sys

def detect_hardware():
    info = {}
    
    # NVIDIA GPU
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=5)
        info["nvidia"] = [line.strip() for line in r.stdout.strip().split('\n') if line.strip()]
    except FileNotFoundError:
        info["nvidia"] = []
    
    # Apple Silicon
    try:
        r = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                          capture_output=True, text=True, timeout=2)
        cpu = r.stdout.strip()
        if "Apple" in cpu:
            info["apple_silicon"] = cpu
            r2 = subprocess.run(["sysctl", "-n", "hw.memsize"],
                               capture_output=True, text=True, timeout=2)
            mem_gb = int(r2.stdout.strip()) / (1024**3)
            info["unified_memory_gb"] = round(mem_gb, 1)
    except:
        pass
    
    return info

hw = detect_hardware()
for k, v in hw.items():
    print(f"{k}: {v}")
    
# 根据硬件推荐学习路径
if hw.get("nvidia"):
    print("\n✅ 有 NVIDIA GPU → 推荐路径: server/ (vLLM, TensorRT-LLM)")
elif hw.get("apple_silicon"):
    print(f"\n✅ Apple Silicon ({hw['unified_memory_gb']}GB) → 推荐路径: local/ (llama.cpp, MLX)")
else:
    print("\n⚠️ 无 GPU → 推荐路径: local/llama.cpp CPU 推理")
